In [ ]:
import pandas as pd
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Step 1: Sample Data
data = {
    "text": [
        "I love this product! It's amazing.",
        "Terrible experience. Never buying again!",
        "Absolutely fantastic! Highly recommend.",
        "Not great. Could be much better.",
        "I'm so happy with this purchase.",
        "Worst item I've ever bought. Awful!"
    ],
    "label": [1, 0, 1, 0, 1, 0]  # 1 = Positive, 0 = Negative
}

# Create a DataFrame
df = pd.DataFrame(data)

# Step 2: Text Preprocessing Function
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Tokenize
    words = text.split()
    # Remove stopwords
    stop_words = set(stopwords.words("english"))
    words = [word for word in words if word not in stop_words]
    # Lemmatize words
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

# Apply preprocessing to the text column
df["processed_text"] = df["text"].apply(preprocess_text)

# Step 3: Feature Extraction using TF-IDF
vectorizer = TfidfVectorizer(max_features=500)  # Limit to top 500 features for performance
X = vectorizer.fit_transform(df["processed_text"])  # Transform text into numerical features
y = df["label"]  # Labels

# Step 4: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Train a Logistic Regression Model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Step 6: Make Predictions
y_pred = model.predict(X_test)

# Step 7: Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

# Step 8: Test the Pipeline on New Data
new_text = "This is an outstanding product. I absolutely love it!"
processed_new_text = preprocess_text(new_text)  # Preprocess the new text
new_features = vectorizer.transform([processed_new_text])  # Convert text to features
prediction = model.predict(new_features)

print(f"Sentiment Prediction for '{new_text}': {'Positive' if prediction[0] == 1 else 'Negative'}")
